In [1]:
import os, sys
sys.path.append('../')

# 10 金融计算模块 (core.financial)

提供常用的金融计算函数。

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from hscredit import init_setting
from hscredit.core.financial import (
    fv, pv, pmt, nper, ipmt, ppmt, rate,
    npv, irr, mirr
)

init_setting()

print("金融计算模块演示")

金融计算模块演示


## 1. 基本金融计算

In [3]:
# 未来值计算
# 每月存1000元，年利率5%，存10年后的未来值
future_value = fv(
    rate=0.05/12,  # 月利率
    nper=10*12,     # 120个月
    pmt=-1000,      # 每月存入
    pv=0            # 初始金额
)
print(f"每月存1000元，10年后未来值: {abs(future_value):,.2f}元")

# 现值计算
# 5年后需要10万元，年利率5%，现在需要存多少钱
present_value = pv(
    rate=0.05,
    nper=5,
    pmt=0,
    fv=-100000
)
print(f"5年后10万元的现值: {abs(present_value):,.2f}元")

# 每期付款额计算
# 贷款100万，年利率6%，期限20年，每月还款额
monthly_payment = pmt(
    rate=0.06/12,
    nper=20*12,
    pv=1000000
)
print(f"100万贷款每月还款额: {abs(monthly_payment):,.2f}元")

每月存1000元，10年后未来值: 155,282.28元
5年后10万元的现值: 78,352.62元
100万贷款每月还款额: 7,164.31元


## 1.1 期数 / 利息本金拆分 / 利率反推

`nper`（期数）、`ipmt`（每期利息）、`ppmt`（每期本金）、`rate`（利率反推）补全基础金融计算的全部 7 个函数。

In [4]:
# 期数 nper: 贷款10万，月利率0.5%，每月还款2000元，需要多少期还清
periods = nper(rate=0.005, pmt=-2000, pv=100000)
print(f"贷款10万每月还2000元需还: {periods:.1f}期 (约{periods/12:.1f}年)")

# 等额本息还款拆分: 贷款100万，月利率0.5%，期限240期(20年)
loan, mrate, n = 1000000, 0.005, 240
monthly = abs(pmt(mrate, n, loan))
print(f"\n贷款100万 / 月利率0.5% / 240期，每月还款: {monthly:,.2f}元")
print(f"{'期数':>4}{'利息(ipmt)':>14}{'本金(ppmt)':>14}{'合计':>14}")
for per in [1, 120, 240]:  # 首期 / 中期 / 末期
    interest = abs(ipmt(mrate, per, n, loan))
    principal = abs(ppmt(mrate, per, n, loan))
    print(f"{per:>4}{interest:>14,.2f}{principal:>14,.2f}{interest+principal:>14,.2f}")

# 利率反推 rate: 贷款10万，分12期，每期还8884.88元，反推月利率
implied_rate = rate(nper=12, pmt=-8884.88, pv=100000)
print(f"\n12期每期还8884.88元反推月利率: {implied_rate:.4%} (年化约{implied_rate*12:.2%})")

贷款10万每月还2000元需还: 57.7期 (约4.8年)

贷款100万 / 月利率0.5% / 240期，每月还款: 7,164.31元
  期数      利息(ipmt)      本金(ppmt)            合计
   1      5,000.00      2,164.31      7,164.31
 120      3,246.16      3,918.15      7,164.31
 240         35.64      7,128.67      7,164.31

12期每期还8884.88元反推月利率: 1.0000% (年化约12.00%)


## 2. 高级金融计算

In [5]:
# NPV计算
cash_flows = [-100000, 30000, 40000, 35000, 25000]  # 初始投资和各期现金流
discount_rate = 0.08
npv_value = npv(discount_rate, cash_flows)
print(f"项目NPV (折现率8%): {npv_value:,.2f}元")

# IRR计算
irr_value = irr(cash_flows)
print(f"项目IRR: {irr_value:.2%}")

# MIRR计算
finance_rate = 0.06  # 融资成本
reinvest_rate = 0.05  # 再投资收益率
mirr_value = mirr(cash_flows, finance_rate, reinvest_rate)
print(f"项目MIRR: {mirr_value:.2%}")

项目NPV (折现率8%): 8,231.21元
项目IRR: 11.74%
项目MIRR: 8.89%


## 3. 风控场景应用

In [6]:
# 场景1: 计算贷款实际年化成本
loan_amount = 100000  # 贷款金额
monthly_rate = 0.01   # 月利率1%
months = 12           # 12期

monthly_pmt = abs(pmt(monthly_rate, months, loan_amount))
total_payment = monthly_pmt * months
total_interest = total_payment - loan_amount

print(f"贷款金额: {loan_amount:,.2f}元")
print(f"月利率: {monthly_rate:.2%}")
print(f"每月还款: {monthly_pmt:,.2f}元")
print(f"总还款额: {total_payment:,.2f}元")
print(f"总利息: {total_interest:,.2f}元")
print(f"实际年化成本: {(total_interest/loan_amount):.2%}")

贷款金额: 100,000.00元
月利率: 1.00%
每月还款: 8,884.88元
总还款额: 106,618.55元
总利息: 6,618.55元
实际年化成本: 6.62%
